CẤU HÌNH MÔI TRƯỜNG (Standard Setup)

In [ ]:
import os
# --- CELL 1: CẤU HÌNH CHUNG ---
STORAGE_ACCOUNT_NAME = "nhatlong"
# DÁN KEY CỦA BẠN VÀO DƯỚI ĐÂY:
STORAGE_ACCOUNT_KEY  = os.environ["AZURE_STORAGE_KEY"]

spark.conf.set(f"fs.azure.account.key.{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net", STORAGE_ACCOUNT_KEY)

# Đường dẫn & Tên bảng
CHECKPOINT_GOLD_PATH = f"abfss://checkpoint@{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net/gold_stats"
TBL_SILVER_SOURCE    = "silver_server_telemetry" # Đọc từ Silver
TBL_GOLD_OUTPUT      = "gold_datacenter_stats"   # Ghi vào Gold

print("✅ Đã kết nối và sẵn sàng cho Giai đoạn Gold.")

TÍNH TOÁN NGHIỆP VỤ (WINDOW AGGREGATION)

In [ ]:
# --- CELL 2: LOGIC TÍNH TOÁN CỬA SỔ (WINDOWING) ---
from pyspark.sql.functions import window, avg, max, min, count, col

# 1. Đọc luồng từ Silver
df_silver = spark.readStream.table(TBL_SILVER_SOURCE)

# 2. Định nghĩa Watermark (Chấp nhận dữ liệu đến muộn tối đa 2 phút)
# Đây là kỹ thuật quan trọng trong Streaming để xử lý độ trễ mạng
df_with_watermark = df_silver.withWatermark("generated_time", "2 minutes")

# 3. Thực hiện Aggregation (Gom nhóm)
df_gold_logic = df_with_watermark.groupBy(
    # Cửa sổ trượt: Độ rộng 1 phút, trượt đi mỗi 10 giây
    window(col("generated_time"), "1 minute", "10 seconds"),
    col("location"),         # Gom theo vị trí (Hanoi, HCM...)
    col("data_center_id")    # Gom theo DC
).agg(
    # Các chỉ số KPI quan trọng
    avg("cpu_usage").alias("avg_cpu"),        # CPU trung bình
    max("cpu_usage").alias("max_cpu"),        # CPU cao nhất (để phát hiện đỉnh tải)
    avg("temperature").alias("avg_temp"),     # Nhiệt độ trung bình
    count("server_id").alias("total_logs")    # Tổng số log nhận được
).select(
    # Làm phẳng kết quả để dễ lưu vào bảng
    col("window.start").alias("window_start"),
    col("window.end").alias("window_end"),
    "location",
    "data_center_id",
    "avg_cpu",
    "max_cpu",
    "avg_temp",
    "total_logs"
)

print("✅ Đã thiết lập xong Logic tính toán Windowing.")

GHI VÀO LỚP GOLD (COMPLETE MODE)

In [ ]:
# --- CELL 3: KÍCH HOẠT LUỒNG GHI VÀO GOLD ---

# Xóa checkpoint cũ nếu cần chạy lại từ đầu
# dbutils.fs.rm(CHECKPOINT_GOLD_PATH, True)

print(f"🚀 Đang tính toán và ghi vào bảng '{TBL_GOLD_OUTPUT}'...")

query_gold = (df_gold_logic.writeStream
    .format("delta")
    .outputMode("complete") # Quan trọng: Chế độ Complete cho Aggregation
    .option("checkpointLocation", CHECKPOINT_GOLD_PATH)
    .trigger(processingTime='10 seconds') # Cập nhật bảng đích mỗi 10s
    .table(TBL_GOLD_OUTPUT)
)

print(f"Stream ID: {query_gold.id}")

TỐI ƯU HÓA (OPTIMIZATION & Z-ORDERING)

In [ ]:
%sql
-- --- CELL 4: TỐI ƯU HÓA BẢNG DELTA (CHẠY THỦ CÔNG) ---
-- Kỹ thuật này giúp gom các file nhỏ thành file lớn và sắp xếp lại dữ liệu

OPTIMIZE gold_datacenter_stats ZORDER BY (location, data_center_id);

-- Sau khi chạy lệnh này, Databricks sẽ báo cáo:
-- "num_files_added", "num_files_removed".
-- Bạn hãy chụp ảnh kết quả này đưa vào báo cáo mục "Tối ưu hóa".

BƯỚC 5: SỬ DỤNG DATASET 3 (ERROR DEFS) ĐỂ TRA CỨU

In [ ]:
%sql
-- --- CELL 5: PHÂN TÍCH LỖI (SỬ DỤNG DATASET 3) ---

-- 1. Tạo View và ÉP KIỂU cột 'details' từ String sang Struct
-- Chúng ta định nghĩa rõ cấu trúc bên trong của JSON để Spark hiểu
CREATE OR REPLACE TEMP VIEW view_error_defs_fixed AS 
SELECT 
    error_code,
    description,
    -- Dùng hàm from_json để biến chuỗi String thành Object có thể truy cập
    from_json(details, 'severity STRING, possible_causes ARRAY<STRING>, resolution_steps ARRAY<STRING>') as details_struct
FROM bronze_error_definitions;

-- 2. Truy vấn báo cáo (Lúc này đã có thể dùng dấu chấm để gọi thuộc tính)
SELECT 
    s.server_id,
    s.temperature,
    s.location,
    e.description AS error_desc,
    e.details_struct.resolution_steps AS huong_dan_xu_ly -- Truy cập vào struct đã parse
FROM silver_server_telemetry s
JOIN view_error_defs_fixed e ON e.error_code = 'ERR_CPU_HIGH' -- Giả lập join lỗi CPU cao
WHERE s.temperature > 50 -- Lấy các server nóng
ORDER BY s.generated_time DESC
LIMIT 5;

BƯỚC 6: KIỂM TRA KẾT QUẢ CUỐI CÙNG

In [ ]:
%sql
-- Xem bảng thống kê Gold (Dữ liệu cho Power BI)
SELECT * FROM gold_datacenter_stats ORDER BY window_end DESC;